In [ ]:
!pip install -q langchain langchain-anthropic

In [ ]:
from google.colab import userdata
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_anthropic import ChatAnthropic
from langchain_core.messages.content import create_image_block, create_text_block
from pydantic import BaseModel, SecretStr

anthropic_api_key = SecretStr(userdata.get('ANTHROPIC_API_KEY'))

def print_response(response: AIMessage):
    print(f"Response id: {response.id}")
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get("input_tokens", 0)
        cached_tokens = response.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        output_tokens = response.usage_metadata.get("output_tokens", 0)
        reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)

## Basic usage

In [ ]:
anthropic_default_model = ChatAnthropic(model="claude-haiku-4-5", api_key=anthropic_api_key)
museum_audio_guide_response = anthropic_default_model.invoke(
    input=[
        SystemMessage("You are a helpful history teaching assistant."),
        HumanMessage("Write a short museum audio-guide introduction for first-time visitors standing in front of the Rosetta Stone. Keep it under 5 sentences.")
    ]
)

In [ ]:
print_response(museum_audio_guide_response)

## Streaming

In [ ]:
for chunk in anthropic_default_model.stream(
    input=[
        SystemMessage("You are an expert in culinary."),
        HumanMessage("Design a one-evening street-food route through Seoul for a curious first-time visitor who wants bold flavors but no seafood.")
    ]
):
    if chunk.text:
        print(chunk.text, end="", flush=True)

## Multi-turn conversations

In [ ]:
conversation = [
    SystemMessage("You are concise and remember user details from the chat history you receive."),
    HumanMessage("My name is Maria. I live in Plovdiv and I am preparing for a Python exam."),
]

first_reply = anthropic_default_model.invoke(conversation)
conversation.append(first_reply)

In [ ]:
print_response(first_reply)

In [ ]:
conversation.append(
    HumanMessage("What do you remember about me, and what should I focus on this week?")
)

second_reply = anthropic_default_model.invoke(conversation)
conversation.append(second_reply)

In [ ]:
print_response(second_reply)

## Reasoning

In [ ]:
anthropic_thinking_model = ChatAnthropic(model="claude-haiku-4-5", api_key=anthropic_api_key, thinking={"type": "enabled", "budget_tokens": 4096})
book_cover_response = anthropic_thinking_model.invoke(
    input=[
        HumanMessage("Write a back-cover blurb for a literary novel about a family-run cinema trying to survive in the streaming era.")
    ]
)

In [ ]:
print_response(book_cover_response)

## Structured output

In [ ]:
class WorkshopBrief(BaseModel):
    title: str
    audience: str
    duration_minutes: float
    key_takeaways: list[str]
    materials_needed: list[str]

anthropic_structured_output_model = anthropic_default_model.with_structured_output(WorkshopBrief)

workshop_brief = anthropic_structured_output_model.invoke(
    input=[
        SystemMessage("You are an expert event organizer."),
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening.")
    ]
)

In [ ]:
workshop_brief

## Vision

<img src="https://freerangestock.com/sample/88947/painter-working-in-studio.jpg" />

In [ ]:
analyze_image_response = anthropic_default_model.invoke(
    input=[
        SystemMessage("You are an expert image analyst. Keep your answer concise and structured."),
        HumanMessage(
            content_blocks=[
              create_text_block("Analyze this image and return a one-sentence summary followed by 5 key visible objects."),
              create_image_block(url="https://freerangestock.com/sample/88947/painter-working-in-studio.jpg")
            ]
        )
    ]
)

In [ ]:
print_response(analyze_image_response)